## 【1】インストール

In [ ]:
# =========================================
# Google Drive 保存設定
# =========================================

import os
from pathlib import Path

SAVE_TO_GOOGLE_DRIVE = True

PROJECT_NAME = "molecular-color-explorer"
DRIVE_PROJECT_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    BASE_DIR = DRIVE_PROJECT_DIR
else:
    BASE_DIR = "."

RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("保存先:", RESULTS_DIR)

In [ ]:
!nvidia-smi

In [ ]:
!pip3 install gpu4pyscf-cuda12x==1.4.3
!pip install cutensor-cu12==1.7.0
!pip install cupy-cuda12x==13.6.0
!pip install rdkit
!pip install pyscf==2.11
!pip install py3Dmol


In [ ]:
!python -c "import pyscf; print('pyscf', pyscf.__version__)"
!python -c "import gpu4pyscf; print('gpu4pyscf', gpu4pyscf.__version__)"
!python -c "import py3Dmol; print('py3Dmol OK')"


## 【2】RDKit
- SMILES → InChIKey
- SMILES → RDKit 3D → XYZ 生成（MMFFで軽く最適化）

In [ ]:
from rdkit import Chem

def smiles_to_inchikey(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.inchi.MolToInchiKey(mol)

In [ ]:
from rdkit.Chem import AllChem

def smiles_to_xyz(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    xyz_lines = []
    for i, atom in enumerate(mol.GetAtoms()):
        p = conf.GetAtomPosition(i)
        xyz_lines.append(f"{atom.GetSymbol()} {p.x:.6f} {p.y:.6f} {p.z:.6f}")
    return "\n".join(xyz_lines)

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def get_structure_info(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    heavy = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() > 1)
    aromatic = rdMolDescriptors.CalcNumAromaticRings(mol)

    return heavy, aromatic

In [ ]:
def structure_check(smiles: str, heavy_limit: int = 17):
    heavy, aromatic = get_structure_info(smiles)

    print(f"SMILES: {smiles}")
    print(f"Heavy atom count     : {heavy}")
    print(f"Aromatic ring count  : {aromatic}")

    if heavy > heavy_limit:
        print(f"⚠️ Heavy atom count が {heavy_limit} を超えています（heavy={heavy}）。計算を中止します。")
        return False, heavy, aromatic

    return True, heavy, aromatic

## 【3】py3Dmol による 3D 構造可視化
- SMILES から RDKit で 3D 構造を作り、py3Dmol で表示する
- TDDFT 計算前に、入力した分子構造が妥当かを目で確認する


In [ ]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_rdkit_molblock(smiles: str, random_seed: int = 0xF00D) -> str:
    """
    SMILES から RDKit で 3D 構造を作り、MolBlock 文字列として返す。
    py3Dmol の addModel(..., "mol") に渡して使う。
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    status = AllChem.EmbedMolecule(mol, params)
    if status != 0:
        # ETKDG で失敗した場合のフォールバック
        status = AllChem.EmbedMolecule(mol, randomSeed=random_seed)

    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        AllChem.UFFOptimizeMolecule(mol)

    return Chem.MolToMolBlock(mol)

def show_molecule_from_smiles(
    smiles: str,
    style: str = "stick",
    width: int = 500,
    height: int = 400,
):
    """
    SMILES を py3Dmol で 3D 表示する。
    style は "stick", "sphere", "line" などを指定可能。
    """
    molblock = smiles_to_rdkit_molblock(smiles)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(molblock, "mol")

    if style == "sphere":
        view.setStyle({"sphere": {"scale": 0.30}})
    elif style == "line":
        view.setStyle({"line": {}})
    else:
        view.setStyle({"stick": {}, "sphere": {"scale": 0.18}})

    view.zoomTo()
    return view.show()

def xyz_lines_to_full_xyz(xyz: str) -> str:
    """
    PySCF に渡している 'C x y z' 形式の複数行 xyz を、
    py3Dmol が読みやすい通常の XYZ 形式に変換する。
    すでに通常の XYZ 形式なら、そのまま返す。
    """
    lines = [line.strip() for line in xyz.strip().splitlines() if line.strip()]
    if len(lines) >= 3 and lines[0].isdigit():
        return xyz

    return f"{len(lines)}\n\n" + "\n".join(lines)

def show_molecule_from_xyz(
    xyz: str,
    style: str = "stick",
    width: int = 500,
    height: int = 400,
):
    """
    smiles_to_xyz() で作った XYZ 文字列を py3Dmol で表示する。
    TDDFT に実際に渡した構造を確認したいときに使う。
    """
    full_xyz = xyz_lines_to_full_xyz(xyz)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(full_xyz, "xyz")

    if style == "sphere":
        view.setStyle({"sphere": {"scale": 0.30}})
    elif style == "line":
        view.setStyle({"line": {}})
    else:
        view.setStyle({"stick": {}, "sphere": {"scale": 0.18}})

    view.zoomTo()
    return view.show()


## 【4】PySCF TDDFT（構造最適化なし・垂直励起のみ）

In [ ]:
import os
import re
import numpy as np
from pyscf import gto, dft, tdscf

HARTREE_TO_EV = 27.211386
EV_TO_NM = 1239.842

def safe_name(text: str) -> str:
    """ファイル名に使いやすい文字列へ変換する。"""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))

def make_calc_tag(
    basis: str,
    xc: str,
    grid_level: int,
    nstates: int,
    charge: int = 0,
) -> str:
    """
    計算条件ごとに結果ファイル名を分けるためのタグ。
    basis や grid を変えても、古い結果と混ざらないようにする。
    """
    return safe_name(f"q{charge}_{xc}_{basis}_grid{grid_level}_nst{nstates}")

def build_pyscf_mol(xyz: str, basis: str, charge: int = 0, verbose: int = 0):
    """電子数から spin を自動判定し、PySCF mol を作る。"""
    mol0 = gto.Mole(atom=xyz, basis=basis, charge=charge, spin=0, unit="Ang")
    mol0.build()
    ne = mol0.nelectron
    spin = 0 if ne % 2 == 0 else 1

    mol = gto.Mole(atom=xyz, basis=basis, charge=charge, spin=spin, unit="Ang", verbose=verbose)
    mol.build()
    return mol, ne, spin

def mo_relative_label(mo_index: int, nocc: int) -> str:
    """
    0始まりのMO番号を HOMO/LUMO 相対表記へ変換する。
    例: HOMO, HOMO-1, LUMO, LUMO+1
    """
    homo = nocc - 1
    lumo = nocc
    if mo_index == homo:
        return "HOMO"
    if mo_index < homo:
        return f"HOMO-{homo - mo_index}"
    if mo_index == lumo:
        return "LUMO"
    return f"LUMO+{mo_index - lumo}"

def _get_td_amplitude_array(xy_entry):
    """
    PySCF TD/TDA の td.xy の1状態分から、X振幅を取り出す。
    TDAではYが0に近い場合が多いので、主な遷移の対応表ではXを使う。
    """
    if isinstance(xy_entry, (tuple, list)):
        x = xy_entry[0]
    else:
        x = xy_entry
    return np.asarray(x)

def extract_transition_table(
    td,
    oscillator_strengths=None,
    coeff_threshold: float = 0.01,
    top_n: int = 8,
):
    """
    TDDFT/TDAの結果から、励起状態ごとの主なMO遷移を表にする。

    出力:
      transition_rows:
        各励起状態の「MO i → MO a」と係数を縦持ちにした詳細表
      state_summary:
        各励起状態ごとに、主要遷移を1セルにまとめた要約表

    注意:
      ここでは PySCF の MO index は 0 始まりで表示する。
      HOMO/LUMO からの相対表記も併記する。
    """
    if not hasattr(td, "xy") or td.xy is None:
        return [], []

    mf = td._scf
    mol = mf.mol

    # 閉殻RKSを主対象にした表示。開殻UKSでは alpha/beta が混ざる場合がある。
    # 教材用途ではまず閉殻一重項の結果を想定する。
    nocc = mol.nelectron // 2
    nmo = mf.mo_coeff.shape[-1] if hasattr(mf.mo_coeff, "shape") else len(mf.mo_occ)
    nvirt = nmo - nocc

    if oscillator_strengths is None:
        try:
            oscillator_strengths = td.oscillator_strength()
        except Exception:
            oscillator_strengths = [None] * len(td.e)

    transition_rows = []
    state_summary = []

    for state_idx, (energy_h, xy_entry) in enumerate(zip(td.e, td.xy), start=1):
        energy_h = float(energy_h)
        energy_ev = energy_h * HARTREE_TO_EV
        wavelength_nm = EV_TO_NM / energy_ev if energy_ev > 0 else None
        osc = None
        if oscillator_strengths is not None and len(oscillator_strengths) >= state_idx:
            try:
                osc = float(oscillator_strengths[state_idx - 1])
            except Exception:
                osc = None

        x = _get_td_amplitude_array(xy_entry)

        # RKS/TDA想定: shape = (nocc, nvirt)
        # 余分な次元がある場合も、最後はフラット化して nocc*nvirt と対応させる。
        flat = np.ravel(x)
        norm = float(np.sum(np.abs(flat) ** 2)) if flat.size > 0 else 0.0

        candidates = []
        for flat_idx, c in enumerate(flat):
            c_real = float(np.real(c))
            if abs(c_real) < coeff_threshold:
                continue

            occ_local = flat_idx // nvirt
            virt_local = flat_idx % nvirt
            occ_mo = int(occ_local)
            virt_mo = int(nocc + virt_local)

            if occ_mo < 0 or occ_mo >= nocc or virt_mo < nocc or virt_mo >= nmo:
                continue

            contribution_percent = 100.0 * (abs(c_real) ** 2) / norm if norm > 0 else None
            candidates.append({
                "state": state_idx,
                "energy_eV": energy_ev,
                "wavelength_nm": wavelength_nm,
                "oscillator_strength": osc,
                "occ_mo": occ_mo,
                "virt_mo": virt_mo,
                "occ_label": mo_relative_label(occ_mo, nocc),
                "virt_label": mo_relative_label(virt_mo, nocc),
                "transition": f"MO {occ_mo} ({mo_relative_label(occ_mo, nocc)}) → MO {virt_mo} ({mo_relative_label(virt_mo, nocc)})",
                "coefficient": c_real,
                "abs_coefficient": abs(c_real),
                "contribution_percent": contribution_percent,
            })

        candidates = sorted(candidates, key=lambda r: r["abs_coefficient"], reverse=True)
        candidates = candidates[:top_n]

        for rank, row in enumerate(candidates, start=1):
            row["rank_in_state"] = rank
            transition_rows.append(row)

        main_text = " / ".join(
            [
                f"{r['transition']} 係数 {r['coefficient']:+.4f}"
                for r in candidates[:3]
            ]
        )
        if not main_text:
            main_text = f"係数 |c| < {coeff_threshold} のため省略"

        state_summary.append({
            "state": state_idx,
            "energy_eV": energy_ev,
            "wavelength_nm": wavelength_nm,
            "oscillator_strength": osc,
            "main_transitions": main_text,
        })

    return transition_rows, state_summary

def tddft_vertical(
    xyz,
    use_gpu=True,
    charge=0,
    basis="def2-TZVP",
    xc="CAM-B3LYP",
    nstates=8,
    scf_grid_level=1,
    chkfile_path=None,
    reuse_chk=True,
    transition_coeff_threshold=0.01,
    transition_top_n=8,
):
    """
    TDDFT 垂直励起計算。

    - chkfile_path を指定すると SCF 結果を chk に保存する。
    - reuse_chk=True かつ chk が存在する場合、SCF 初期値として chk を使う。
      完全に同じ条件の JSON がある場合は、save_result_with_png 側で TDDFT 全体をスキップする。
    - scf_grid_level で DFT grid の粗さを指定できる。
    - td.xy から励起状態ごとの主な MO 遷移表も作成する。
    """
    mol, ne, spin = build_pyscf_mol(xyz, basis=basis, charge=charge, verbose=0)

    # DFT（密度フィッティング）
    if spin == 0:
        mf = dft.RKS(mol).density_fit()
    else:
        mf = dft.UKS(mol).density_fit()

    mf.xc = xc
    mf.grids.level = scf_grid_level

    if chkfile_path is not None:
        os.makedirs(os.path.dirname(chkfile_path), exist_ok=True)
        mf.chkfile = chkfile_path
        if reuse_chk and os.path.exists(chkfile_path):
            print(f"既存 chk を SCF 初期値として使用: {chkfile_path}")
            mf.init_guess = "chkfile"

    # ==== ここで GPU へ移行 ====
    if use_gpu:
        try:
            mf = mf.to_gpu()
            if chkfile_path is not None:
                mf.chkfile = chkfile_path
        except AttributeError:
            pass

    mf.kernel()

    # TDA
    if spin == 0:
        td = mf.TDA()
    else:
        td = tdscf.UKS(mf)
    td.nstates = nstates

    # ==== TD も GPU に載せられる ====
    if use_gpu:
        try:
            td = td.to_gpu()
        except AttributeError:
            pass

    td.kernel()

    e_hartree = np.asarray(td.e)
    e_ev = e_hartree * HARTREE_TO_EV
    wl_nm = EV_TO_NM / e_ev

    try:
        osc = td.oscillator_strength()
    except Exception:
        osc = np.zeros_like(wl_nm)

    transition_rows, state_summary = extract_transition_table(
        td,
        oscillator_strengths=osc,
        coeff_threshold=transition_coeff_threshold,
        top_n=transition_top_n,
    )

    return (
        wl_nm.tolist(),
        np.asarray(osc).tolist(),
        ne,
        spin,
        e_hartree.tolist(),
        e_ev.tolist(),
        transition_rows,
        state_summary,
    )

## 【5】分子軌道の可視化（任意）
- `chk` ファイルから SCF 結果を読み込み、再SCFせずに軌道 cube を作成できる
- `HOMO`, `LUMO` だけでなく、`HOMO-1`, `HOMO-2`, `LUMO+1` のように表示する軌道を選べる
- TDDFT と同じ basis の chk を使うか、可視化用に軽い basis で別途 SCF するかを選べる


In [ ]:
import os
import pandas as pd
from pyscf import tools, lib
from pyscf.scf import chkfile as scf_chkfile

def mol_to_xyz_for_py3dmol(mol) -> str:
    """PySCF の mol オブジェクトを py3Dmol 用の XYZ 形式に変換する。"""
    xyz = f"{mol.natm}\n\n"
    coords = mol.atom_coords(unit="Angstrom")
    for i in range(mol.natm):
        sym = mol.atom_symbol(i)
        x, y, z = coords[i]
        xyz += f"{sym} {x:.6f} {y:.6f} {z:.6f}\n"
    return xyz

def calc_ground_state_for_orbitals(
    xyz: str,
    charge: int = 0,
    basis: str = "6-31G*",
    xc: str = "B3LYP",
    scf_grid_level: int = 1,
    chkfile_path: str | None = None,
    reuse_chk: bool = True,
):
    """
    軌道可視化用の基底状態DFT計算を行う。
    chkfile_path を指定すると保存・再利用できる。
    """
    mol, ne, spin = build_pyscf_mol(xyz, basis=basis, charge=charge, verbose=0)

    if spin == 0:
        mf = dft.RKS(mol)
    else:
        mf = dft.UKS(mol)

    mf.xc = xc
    mf.grids.level = scf_grid_level

    if chkfile_path is not None:
        os.makedirs(os.path.dirname(chkfile_path), exist_ok=True)
        mf.chkfile = chkfile_path
        if reuse_chk and os.path.exists(chkfile_path):
            print(f"既存 chk を SCF 初期値として使用: {chkfile_path}")
            mf.init_guess = "chkfile"

    mf.kernel()
    return mol, mf, ne, spin

def load_orbitals_from_chk(chkfile_path: str):
    """
    PySCF chk から mol, mo_coeff, mo_occ, mo_energy を読み込む。
    TDDFT 計算時に保存された chk を使えば、軌道表示のための SCF 再計算を避けられる。
    """
    if not os.path.exists(chkfile_path):
        raise FileNotFoundError(f"chk file not found: {chkfile_path}")

    mol = scf_chkfile.load_mol(chkfile_path)
    scf_data = lib.chkfile.load(chkfile_path, "scf")
    mo_coeff = scf_data["mo_coeff"]
    mo_occ = scf_data["mo_occ"]
    mo_energy = scf_data.get("mo_energy", None)
    return mol, mo_coeff, mo_occ, mo_energy

def pick_spin_channel(mo_coeff, mo_occ, mo_energy=None, spin_channel: str = "alpha"):
    """
    RKS/RHF ならそのまま、UKS/UHF なら alpha または beta の軌道を取り出す。
    """
    if isinstance(mo_coeff, (list, tuple)) or getattr(mo_coeff, "ndim", 2) == 3:
        idx = 0 if spin_channel.lower() in ["alpha", "a", "0"] else 1
        coeff = mo_coeff[idx]
        occ = mo_occ[idx]
        energy = None if mo_energy is None else mo_energy[idx]
    else:
        coeff = mo_coeff
        occ = mo_occ
        energy = mo_energy
    return coeff, occ, energy

def resolve_orbital_index(label, mo_occ, nmo: int):
    """
    軌道ラベルを MO index に変換する。

    例:
      "HOMO"   -> HOMO
      "HOMO-1" -> HOMO の1つ下
      "LUMO"   -> LUMO
      "LUMO+2" -> LUMO の2つ上
      5        -> MO index 5（0始まり）
    """
    if isinstance(label, int):
        idx = label
        if idx < 0 or idx >= nmo:
            raise ValueError(f"orbital index out of range: {idx}")
        return idx, f"MO{idx}"

    text = str(label).upper().replace(" ", "")
    occ_indices = [i for i, occ in enumerate(mo_occ) if occ > 1e-6]
    if not occ_indices:
        raise ValueError("占有軌道が見つかりません。mo_occ を確認してください。")

    homo = max(occ_indices)
    lumo = homo + 1

    if text == "HOMO":
        idx = homo
    elif text.startswith("HOMO-"):
        idx = homo - int(text.split("-")[1])
    elif text == "LUMO":
        idx = lumo
    elif text.startswith("LUMO+"):
        idx = lumo + int(text.split("+")[1])
    elif text.startswith("MO"):
        idx = int(text.replace("MO", ""))
    else:
        raise ValueError(f"軌道指定を解釈できません: {label}")

    if idx < 0 or idx >= nmo:
        raise ValueError(f"{label} -> MO index {idx} は範囲外です（0〜{nmo-1}）。")

    return idx, text

def save_selected_orbital_cubes(
    xyz: str,
    out_dir: str,
    orbital_labels=("HOMO", "LUMO"),
    prefix: str = "orbital",
    charge: int = 0,
    basis: str = "6-31G*",
    xc: str = "B3LYP",
    scf_grid_level: int = 1,
    cube_grid: int = 80,
    chkfile_path: str | None = None,
    use_chk_orbitals: bool = True,
    orbital_chkfile_path: str | None = None,
    spin_channel: str = "alpha",
):
    """
    指定した軌道の cube ファイルを保存する。

    use_chk_orbitals=True かつ chkfile_path が存在する場合:
        chk から軌道を読み込み、SCF を再計算しない。
    それ以外:
        可視化用 basis/xc/grid で SCF を実行し、必要なら orbital_chkfile_path に保存する。
    """
    os.makedirs(out_dir, exist_ok=True)

    if use_chk_orbitals and chkfile_path is not None and os.path.exists(chkfile_path):
        print(f"chk から軌道を読み込みます（SCF再計算なし）: {chkfile_path}")
        mol, mo_coeff, mo_occ, mo_energy = load_orbitals_from_chk(chkfile_path)
        source = "chk"
    else:
        print("可視化用の SCF を実行します。")
        mol, mf, ne, spin = calc_ground_state_for_orbitals(
            xyz=xyz,
            charge=charge,
            basis=basis,
            xc=xc,
            scf_grid_level=scf_grid_level,
            chkfile_path=orbital_chkfile_path,
            reuse_chk=True,
        )
        mo_coeff = mf.mo_coeff
        mo_occ = mf.mo_occ
        mo_energy = getattr(mf, "mo_energy", None)
        source = "new_scf"

    coeff, occ, energy = pick_spin_channel(mo_coeff, mo_occ, mo_energy, spin_channel=spin_channel)
    nmo = coeff.shape[1]

    cube_paths = {}
    info_rows = []

    for label in orbital_labels:
        idx, normalized_label = resolve_orbital_index(label, occ, nmo)
        cube_name = f"{prefix}_{safe_name(normalized_label)}.cube"
        cube_path = os.path.join(out_dir, cube_name)
        tools.cubegen.orbital(mol, cube_path, coeff[:, idx], nx=cube_grid, ny=cube_grid, nz=cube_grid)
        cube_paths[normalized_label] = cube_path

        info_rows.append({
            "label": normalized_label,
            "mo_index_0_based": idx,
            "occupancy": float(occ[idx]),
            "energy_hartree": None if energy is None else float(energy[idx]),
            "cube_file": cube_path,
            "source": source,
        })
        print(f"Saved {normalized_label}: {cube_path}")

    orbital_info = pd.DataFrame(info_rows)
    return mol, cube_paths, orbital_info

def show_orbital_py3dmol(
    mol,
    cube_file: str,
    isoval: float = 0.03,
    width: int = 500,
    height: int = 400,
):
    """
    cube ファイルを py3Dmol で表示する。
    正の位相を blue、負の位相を red で表示。
    """
    with open(cube_file, "r") as f:
        cube_data = f.read()

    view = py3Dmol.view(width=width, height=height)
    view.addModel(mol_to_xyz_for_py3dmol(mol), "xyz")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.15}})

    view.addVolumetricData(
        cube_data, "cube",
        {"isoval": isoval, "color": "blue", "opacity": 0.65}
    )
    view.addVolumetricData(
        cube_data, "cube",
        {"isoval": -isoval, "color": "red", "opacity": 0.65}
    )

    view.zoomTo()
    return view.show()


## 【6】電子状態の解析と結果の整理

### 禁制遷移を除外する（しきい値 f < 0.01）

In [ ]:
def filter_forbidden_transitions(wavelengths_nm, osc_strength, threshold=0.01):
    wl2 = []
    f2 = []
    for wl, f in zip(wavelengths_nm, osc_strength):
        if f >= threshold:
            wl2.append(float(wl))
            f2.append(float(f))
    return wl2, f2


### **スペクトル** PNG を保存する関数

In [ ]:
import matplotlib.pyplot as plt

def save_uv_spectrum_png(wavelengths_nm, osc_strength, png_path,
                         fwhm=15, resolution=0.2,
                         wl_min=100, wl_max=800):

    wl_grid = np.arange(wl_min, wl_max, resolution)
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    spectrum = np.zeros_like(wl_grid)

    for wl, f in zip(wavelengths_nm, osc_strength):
        spectrum += f * np.exp(-(wl_grid - wl)**2 / (2 * sigma**2))

    plt.figure(figsize=(8,4))
    plt.plot(wl_grid, spectrum, lw=2)
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Intensity (a.u.)")
    plt.title("Simulated UV-Vis Spectrum")
    plt.xlim(wl_min, wl_max)
    plt.tight_layout()
    plt.savefig(png_path, dpi=200)
    plt.close()

In [ ]:
import os
import json
import numpy as np
import pandas as pd

def summarize_results(base_dir=RESULTS_DIR):
    """
    results/ の中にある InChIKey フォルダを走査し、
    各 JSON から以下を抽出して DataFrame にまとめる：

    - smiles
    - allowed_wavelength_nm の最大値（最大波長）
    - そのときの allowed_oscillator_strength
    - inchikey（フォルダ名 or JSON 内）
    - json_file（ファイル名）
    """
    rows = []

    for inchikey in os.listdir(base_dir):
        folder = os.path.join(base_dir, inchikey)
        if not os.path.isdir(folder):
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".json"):
                continue

            json_path = os.path.join(folder, fname)
            with open(json_path, "r") as f:
                data = json.load(f)

            smiles = data.get("smiles")
            wl = data.get("allowed_wavelength_nm") or []
            osc = data.get("allowed_oscillator_strength") or []

            # allowed が空なら skip
            if not wl or not osc:
                continue

            wl_arr = np.array(wl, dtype=float)
            osc_arr = np.array(osc, dtype=float)

            # 最も長波長の遷移を採用
            idx = int(np.argmax(wl_arr))

            rows.append({
                "inchikey": data.get("inchikey", inchikey),
                "smiles": smiles,
                "lambda_max_nm": wl_arr[idx],
                "osc_at_lambda_max": osc_arr[idx],
            })

    df = pd.DataFrame(rows)

    # λmax の降順にして見やすく
    if not df.empty:
        df = df.sort_values("lambda_max_nm", ascending=False).reset_index(drop=True)

    return df


## 【7】2次元マップ化

In [ ]:
import os
import json
import numpy as np
import pandas as pd

def summarize_results(base_dir=RESULTS_DIR):
    """
    results/ 以下の InChIKey フォルダにある JSON を走査して、
    各 JSON から以下を抽出し DataFrame にまとめる：

      - smiles
      - lambda_max_nm（allowed の最大波長）
      - osc_at_lambda_max
      - compute_time_sec（計算時間）
    """
    rows = []

    for inchikey in os.listdir(base_dir):
        folder = os.path.join(base_dir, inchikey)
        if not os.path.isdir(folder):
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".json"):
                continue

            json_path = os.path.join(folder, fname)
            with open(json_path, "r") as f:
                data = json.load(f)

            smiles = data.get("smiles")
            wl = data.get("allowed_wavelength_nm") or []
            osc = data.get("allowed_oscillator_strength") or []
            compute_time = data.get("compute_time_sec", None)   # ← 追加

            # allowed_wavelength_nm が空ならスキップ
            if not wl or not osc:
                continue

            wl_arr = np.array(wl, dtype=float)
            osc_arr = np.array(osc, dtype=float)

            # 最長波長のインデックス
            idx = int(np.argmax(wl_arr))

            rows.append({
                "inchikey": data.get("inchikey", inchikey),
                "json_file": fname,
                "smiles": smiles,
                "lambda_max_nm": wl_arr[idx],
                "osc_at_lambda_max": osc_arr[idx],
                "compute_time_sec": compute_time,   # ← 追加
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("lambda_max_nm", ascending=False).reset_index(drop=True)

    return df

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def calc_heavy_atom_count(smiles: str) -> int:
    """H を除いた原子数（heavy atoms）を返す。"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    return sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() > 1)

def calc_aromatic_ring_count(smiles: str) -> int:
    """芳香環の数を返す（RDKit の aromatic ring 数）。"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    return rdMolDescriptors.CalcNumAromaticRings(mol)

def add_structure_axes(df: pd.DataFrame) -> pd.DataFrame:
    """
    DataFrame に
      - heavy_atom_count（横軸）
      - aromatic_ring_count（縦軸）
    の列を追加して返す。
    """
    df = df.copy()
    df["heavy_atom_count"] = df["smiles"].apply(calc_heavy_atom_count)
    df["aromatic_ring_count"] = df["smiles"].apply(calc_aromatic_ring_count)
    # 計算失敗などを除外
    df = df.dropna(subset=["heavy_atom_count", "aromatic_ring_count", "lambda_max_nm"])
    return df


In [ ]:
import matplotlib.pyplot as plt

def plot_structure_map(df: pd.DataFrame, out_png: str | None = None):
    """
    横軸: heavy_atom_count
    縦軸: aromatic_ring_count
    色  : lambda_max_nm
    の 2D マップをプロット。
    out_png を指定するとファイルとして保存。
    """
    if df.empty:
        print("DataFrame が空です。先に summarize_results を確認してください。")
        return

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        df["heavy_atom_count"],
        df["aromatic_ring_count"],
        c=df["lambda_max_nm"],
        cmap="jet",
        s=60,
        edgecolors="k",
        linewidths=0.5,
    )
    cbar = plt.colorbar(sc)
    cbar.set_label("λmax (nm)")

    plt.xlabel("Heavy atom count")
    plt.ylabel("Aromatic ring count")
    plt.title("Structure Map colored by λmax")

    plt.grid(alpha=0.3)
    plt.tight_layout()

    if out_png is not None:
        plt.savefig(out_png, dpi=200)
        print(f"Saved structure map to: {out_png}")
    plt.show()


In [ ]:
def build_structure_map(base_dir=RESULTS_DIR, out_png="structure_map.png"):
    """
    1. results/ 以下の JSON を集計して λmax を取り出し
    2. SMILES から heavy_atom_count / aromatic_ring_count を計算
    3. 2D マップを描き、PNG も保存
    4. 最終的な DataFrame を返す
    """
    df = summarize_results(base_dir=base_dir)
    if df.empty:
        print("No data found under:", base_dir)
        return df

    df = add_structure_axes(df)
    if df.empty:
        print("構造情報の計算に失敗しました。SMILES を確認してください。")
        return df

    plot_structure_map(df, out_png=out_png)
    return df


## 【8】ワークフロー
- SMILES、basis、functional、grid などのパラメーターをまとめて指定する
- 同じ条件の JSON がすでにあれば TDDFT 計算をスキップする
- `chk` があれば SCF 初期値または軌道可視化に再利用する


In [ ]:
import os
import json
import time

def save_result_with_png(
    smiles,
    use_gpu=True,
    charge=0,
    basis="def2-TZVP",
    xc="CAM-B3LYP",
    nstates=8,
    scf_grid_level=1,
    reuse_saved_result=True,
    reuse_chk=True,
    transition_coeff_threshold=0.01,
    transition_top_n=8,
):
    """
    SMILES から TDDFT 吸収波長を計算し、JSON/PNG/chk/XYZ/遷移対応表を保存する。

    reuse_saved_result=True:
        同じ計算条件の JSON がすでにある場合、TDDFT 計算全体をスキップする。
    reuse_chk=True:
        chk がある場合、SCF の初期値として使う。
    transition_coeff_threshold:
        遷移対応表に表示する係数 |c| のしきい値。
    transition_top_n:
        各励起状態で保存する主な遷移の最大数。
    """
    start_time = time.time()

    inchikey = smiles_to_inchikey(smiles)
    calc_tag = make_calc_tag(basis=basis, xc=xc, grid_level=scf_grid_level, nstates=nstates, charge=charge)

    folder = os.path.join(RESULTS_DIR, inchikey)
    os.makedirs(folder, exist_ok=True)

    xyz = smiles_to_xyz(smiles)

    xyz_path = f"{folder}/{inchikey}.xyz"
    json_path = f"{folder}/{inchikey}_{calc_tag}.json"
    png_path = f"{folder}/{inchikey}_{calc_tag}.png"
    chkfile_path = f"{folder}/{inchikey}_{calc_tag}.chk"
    transition_csv_path = f"{folder}/{inchikey}_{calc_tag}_transitions.csv"
    transition_summary_csv_path = f"{folder}/{inchikey}_{calc_tag}_transition_summary.csv"

    # 同じ条件の結果があれば、TDDFT を丸ごとスキップ
    if reuse_saved_result and os.path.exists(json_path):
        print(f"既存の結果 JSON を読み込みます（TDDFT再計算なし）: {json_path}")
        with open(json_path, "r") as f:
            result = json.load(f)
        # 古い JSON に情報が無い場合の補完
        result.setdefault("chkfile", chkfile_path)
        result.setdefault("basis", basis)
        result.setdefault("xc", xc)
        result.setdefault("scf_grid_level", scf_grid_level)
        result.setdefault("nstates", nstates)
        result.setdefault("transition_csv_file", transition_csv_path)
        result.setdefault("transition_summary_csv_file", transition_summary_csv_path)
        return result, json_path, png_path

    with open(xyz_path, "w") as f:
        f.write(xyz_lines_to_full_xyz(xyz))

    (
        wl_all,
        f_all,
        ne,
        spin,
        energy_hartree,
        energy_ev,
        transition_rows,
        transition_summary,
    ) = tddft_vertical(
        xyz,
        use_gpu=use_gpu,
        charge=charge,
        basis=basis,
        xc=xc,
        nstates=nstates,
        scf_grid_level=scf_grid_level,
        chkfile_path=chkfile_path,
        reuse_chk=reuse_chk,
        transition_coeff_threshold=transition_coeff_threshold,
        transition_top_n=transition_top_n,
    )

    wl_allowed, f_allowed = filter_forbidden_transitions(wl_all, f_all)

    if len(f_allowed) > 0:
        idx = int(np.argmax(f_allowed))
        lambda_max = wl_allowed[idx]
    else:
        lambda_max = None

    elapsed_sec = float(time.time() - start_time)
    print(f"計算時間: {elapsed_sec:.2f} 秒")

    result = {
        "smiles": smiles,
        "inchikey": inchikey,
        "basis": basis,
        "xc": xc,
        "charge": charge,
        "nstates": nstates,
        "scf_grid_level": scf_grid_level,
        "electron_count": ne,
        "spin": spin,
        "lambda_max_nm": lambda_max,
        "all_energy_hartree": energy_hartree,
        "all_energy_eV": energy_ev,
        "all_wavelength_nm": wl_all,
        "oscillator_strength": f_all,
        "allowed_wavelength_nm": wl_allowed,
        "allowed_oscillator_strength": f_allowed,
        "transition_coeff_threshold": transition_coeff_threshold,
        "transition_top_n": transition_top_n,
        "transition_details": transition_rows,
        "transition_summary": transition_summary,
        "transition_csv_file": transition_csv_path,
        "transition_summary_csv_file": transition_summary_csv_path,
        "compute_time_sec": elapsed_sec,
        "xyz_file": xyz_path,
        "chkfile": chkfile_path,
    }

    with open(json_path, "w") as f:
        json.dump(result, f, indent=2)

    # 遷移対応表をCSVにも保存
    if len(transition_rows) > 0:
        pd.DataFrame(transition_rows).to_csv(transition_csv_path, index=False)
        print(f"Saved transition CSV: {transition_csv_path}")
    if len(transition_summary) > 0:
        pd.DataFrame(transition_summary).to_csv(transition_summary_csv_path, index=False)
        print(f"Saved transition summary CSV: {transition_summary_csv_path}")

    save_uv_spectrum_png(
        wl_allowed if len(wl_allowed) > 0 else wl_all,
        f_allowed if len(f_allowed) > 0 else f_all,
        png_path
    )

    print(f"Saved JSON: {json_path}")
    print(f"Saved PNG:  {png_path}")
    print(f"Saved chk:  {chkfile_path}")

    return result, json_path, png_path

## 入力・計算パラメーター
ここを変更すると、basis、functional、grid、表示する軌道をまとめて変えられます。

In [ ]:
# ===== 入力分子 =====
input_SMILES = "C=O"

# ===== TDDFT 計算パラメーター =====
CHARGE = 0
USE_GPU = True
CALC_BASIS = "def2-TZVP"
CALC_XC = "CAM-B3LYP"
N_STATES = 8
SCF_GRID_LEVEL = 1

# ===== 遷移対応表パラメーター =====
# TDDFT の係数 |c| がこの値以上の MO 遷移を表に出す
TRANSITION_COEFF_THRESHOLD = 0.01
# 各励起状態で表示・保存する主な遷移の最大数
TRANSITION_TOP_N = 8

# True: 同じ条件の JSON があれば TDDFT を実行しない
REUSE_SAVED_RESULT = True

# True: chk があれば SCF 初期値として使う
REUSE_CHK = True

# ===== 軌道可視化パラメーター =====
# "td_chk"  : TDDFT計算で保存した chk から軌道を読む（同じ basis、再SCFなし）
# "new_scf" : 可視化用に basis/xc/grid を指定して別途SCFする
ORBITAL_SOURCE = "td_chk"

ORBITAL_BASIS = CALC_BASIS      # 例: "6-31G*" にすると軽い可視化用SCFにできる
ORBITAL_XC = CALC_XC            # 例: "B3LYP"
ORBITAL_GRID_LEVEL = SCF_GRID_LEVEL
CUBE_GRID = 60                  # 大きいほど滑らか。ただし重くなる。例: 40, 60, 80
ISOVAL = 0.03

# 表示したい軌道を指定。例: ["HOMO-2", "HOMO-1", "HOMO", "LUMO", "LUMO+1"]
ORBITALS_TO_SHOW = ["HOMO", "LUMO"]

# パラメーターの確認表示
pd.DataFrame([
    {"parameter": "input_SMILES", "value": input_SMILES},
    {"parameter": "CALC_BASIS", "value": CALC_BASIS},
    {"parameter": "CALC_XC", "value": CALC_XC},
    {"parameter": "N_STATES", "value": N_STATES},
    {"parameter": "SCF_GRID_LEVEL", "value": SCF_GRID_LEVEL},
    {"parameter": "TRANSITION_COEFF_THRESHOLD", "value": TRANSITION_COEFF_THRESHOLD},
    {"parameter": "TRANSITION_TOP_N", "value": TRANSITION_TOP_N},
    {"parameter": "ORBITAL_SOURCE", "value": ORBITAL_SOURCE},
    {"parameter": "ORBITAL_BASIS", "value": ORBITAL_BASIS},
    {"parameter": "CUBE_GRID", "value": CUBE_GRID},
    {"parameter": "ORBITALS_TO_SHOW", "value": ", ".join(ORBITALS_TO_SHOW)},
])


## 吸収波長を計算する部分
同じ SMILES・basis・functional・grid・励起状態数の JSON がすでにある場合は、`REUSE_SAVED_RESULT=True` により再計算をスキップします。

In [ ]:
# ① 構造チェック（heavy atom count / aromatic ring count の表示）
ok, heavy, aromatic = structure_check(input_SMILES)

# ② TDDFT 計算前に、RDKit で作った 3D 構造を確認
if ok:
    show_molecule_from_smiles(input_SMILES)

# ③ heavy atom count に問題なければ計算を実行
if ok:
    result, json_path, png_path = save_result_with_png(
        input_SMILES,
        use_gpu=USE_GPU,
        charge=CHARGE,
        basis=CALC_BASIS,
        xc=CALC_XC,
        nstates=N_STATES,
        scf_grid_level=SCF_GRID_LEVEL,
        reuse_saved_result=REUSE_SAVED_RESULT,
        reuse_chk=REUSE_CHK,
        transition_coeff_threshold=TRANSITION_COEFF_THRESHOLD,
        transition_top_n=TRANSITION_TOP_N,
    )
    result


### 励起状態とMO遷移の対応表を表示する
各励起状態について、エネルギー、波長、振動子強度、主な `MO → MO` 遷移を表にします。  
`TRANSITION_COEFF_THRESHOLD` を小さくすると、より小さい係数の遷移まで表示できます。

In [ ]:
# 励起状態ごとの要約表
if "result" not in globals():
    print("先に TDDFT 計算セルを実行してください。")
else:
    transition_summary = result.get("transition_summary", [])
    transition_details = result.get("transition_details", [])

    if len(transition_summary) == 0:
        print("このJSONには遷移対応表が保存されていません。")
        print("REUSE_SAVED_RESULT = False にして再計算すると、td.xy から遷移対応表を作成できます。")
    else:
        df_transition_summary = pd.DataFrame(transition_summary)
        display(
            df_transition_summary[
                ["state", "energy_eV", "wavelength_nm", "oscillator_strength", "main_transitions"]
            ].style.format({
                "energy_eV": "{:.2f}",
                "wavelength_nm": "{:.1f}",
                "oscillator_strength": "{:.4f}",
            })
        )

        # 代表的な最大吸収に対応する状態を目立たせるため、f が最大の状態を表示
        if "oscillator_strength" in df_transition_summary.columns:
            df_tmp = df_transition_summary.dropna(subset=["oscillator_strength"])
            if not df_tmp.empty:
                idx = df_tmp["oscillator_strength"].astype(float).idxmax()
                row = df_transition_summary.loc[idx]
                print(
                    f"最大の振動子強度をもつ状態: State {int(row['state'])}, "
                    f"{row['energy_eV']:.2f} eV, {row['wavelength_nm']:.1f} nm, "
                    f"f = {row['oscillator_strength']:.4f}"
                )

# 詳細表：1つの励起状態に複数のMO遷移が含まれる場合はこちらを見る
if "result" in globals() and len(result.get("transition_details", [])) > 0:
    df_transition_details = pd.DataFrame(result["transition_details"])
    display(
        df_transition_details[
            [
                "state",
                "rank_in_state",
                "transition",
                "coefficient",
                "contribution_percent",
                "energy_eV",
                "wavelength_nm",
                "oscillator_strength",
            ]
        ].style.format({
            "coefficient": "{:+.4f}",
            "contribution_percent": "{:.1f}",
            "energy_eV": "{:.2f}",
            "wavelength_nm": "{:.1f}",
            "oscillator_strength": "{:.4f}",
        })
    )

    # CSV保存パスも確認
    print("Transition CSV:", result.get("transition_csv_file"))
    print("Transition summary CSV:", result.get("transition_summary_csv_file"))

### 選択した分子軌道を表示する場合（任意）
`ORBITALS_TO_SHOW` に `HOMO-2`, `HOMO-1`, `HOMO`, `LUMO`, `LUMO+1` などを入れると、複数の軌道を選んで表示できます。

- `ORBITAL_SOURCE="td_chk"`: TDDFT 計算で保存した `chk` から軌道を読み込むため、SCF は再計算しません。
- `ORBITAL_SOURCE="new_scf"`: 可視化用に `ORBITAL_BASIS` や `ORBITAL_GRID_LEVEL` を変えて、軽い条件でSCFします。


In [ ]:
# 選択した軌道を表示したい場合は、このセルを実行する

xyz = smiles_to_xyz(input_SMILES)
inchikey = smiles_to_inchikey(input_SMILES)
calc_tag = make_calc_tag(
    basis=CALC_BASIS,
    xc=CALC_XC,
    grid_level=SCF_GRID_LEVEL,
    nstates=N_STATES,
    charge=CHARGE,
)

orbital_dir = f"results/{inchikey}/orbitals_{calc_tag}"

# TDDFT chk を使う場合。new_scf の場合は使わない。
td_chkfile = result.get("chkfile") if ORBITAL_SOURCE == "td_chk" else None

# new_scf 用の chk。ORBITAL_BASIS などを変えた場合はこちらに保存される。
orbital_calc_tag = make_calc_tag(
    basis=ORBITAL_BASIS,
    xc=ORBITAL_XC,
    grid_level=ORBITAL_GRID_LEVEL,
    nstates=0,
    charge=CHARGE,
)
orbital_chkfile = f"results/{inchikey}/{inchikey}_orbital_{orbital_calc_tag}.chk"

mol_vis, cube_paths, orbital_info = save_selected_orbital_cubes(
    xyz=xyz,
    out_dir=orbital_dir,
    orbital_labels=ORBITALS_TO_SHOW,
    prefix=inchikey,
    charge=CHARGE,
    basis=ORBITAL_BASIS,
    xc=ORBITAL_XC,
    scf_grid_level=ORBITAL_GRID_LEVEL,
    cube_grid=CUBE_GRID,
    chkfile_path=td_chkfile,
    use_chk_orbitals=(ORBITAL_SOURCE == "td_chk"),
    orbital_chkfile_path=orbital_chkfile,
    spin_channel="alpha",
)

orbital_info


In [ ]:
# 1つずつ軌道を表示する
# Colabでは複数の3D表示が縦に並びます。

for label, cube_file in cube_paths.items():
    print(label, cube_file)
    show_orbital_py3dmol(mol_vis, cube_file, isoval=ISOVAL)


# resultsフォルダに入っているデータを確認する

In [ ]:
df = summarize_results(RESULTS_DIR)

from rdkit.Chem import PandasTools
PandasTools.AddMoleculeColumnToFrame(df, smilesCol="smiles", molCol="Mol")
df

In [ ]:
df_map = build_structure_map(base_dir=RESULTS_DIR, out_png="structure_map.png")
df_map.head()

# データをレポート用にCSVファイルに保存する

In [ ]:
# ====================================================
# レポート用：不要なカラムを除いてCSVを保存
# ====================================================

from google.colab import files

csv_path = RESULTS_DIR  + "/report_results.csv"

# json_file と Mol が存在する場合だけ削除
df_report = df.drop(columns=["json_file", "Mol"], errors="ignore")

df_report.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"CSV file saved: {csv_path}")